In [1]:
# use bge-base-en-v1.5 in sentence transformer for vectorize each chunks
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
import json

In [ ]:
# from pathlib import Path
# from google.colab import drive
# drive.mount('/content/drive', force_remount=False)
# print('✅ Google Drive mounted at /content/drive')

# # ── Verify T4 GPU ─────────────────────────────────────────────────────────────
# import torch
# assert torch.cuda.is_available(), (
#     '❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU and re-run.'
# )
# gpu_name = torch.cuda.get_device_name(0)
# vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
# print(f'✅ GPU : {gpu_name}')
# print(f'✅ VRAM: {vram_gb:.1f} GB')
# assert vram_gb > 12, f'❌ Expected T4 (16 GB), got {vram_gb:.1f} GB — check runtime.'
# print('✅ T4 confirmed — good to go!')

Mounted at /content/drive
✅ Google Drive mounted at /content/drive
✅ GPU : Tesla T4
✅ VRAM: 15.6 GB
✅ T4 confirmed — good to go!


In [ ]:
# DRIVE_ROOT = Path('/content/drive/MyDrive')


# DATA_DIR   = DRIVE_ROOT / 'data'
# jsonl_folder_path= str(DATA_DIR / 'chunks_jsonl')
# vector_embed_folder_path   = str(DATA_DIR / 'vector_embeddings')


In [ ]:
jsonl_folder_path = "../../data/chunks_jsonl"
vector_embed_folder_path = "../../data/vector_embeddings"


In [4]:
model = SentenceTransformer("BAAI/bge-base-en-v1.5")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [6]:
example_text = "Global Workspace Theory posits that highly complex networks engender perceptual awareness."
embedding = model.encode(example_text)
print("Vector shape (number of dimensions):", embedding.shape)  # Will print (768,)
print("Preview of numbers:", embedding[:5])

Vector shape (number of dimensions): (768,)
Preview of numbers: [0.03530781 0.01851509 0.03937693 0.02639334 0.02954359]


In [ ]:

os.makedirs(vector_embed_folder_path, exist_ok=True)

BATCH_SIZE = 32

for file_name in os.listdir(jsonl_folder_path):
    if file_name.endswith('.jsonl'):
        vector_embed_file_name = file_name.replace('.jsonl', '_vector.jsonl')
        vector_embed_file_path = os.path.join(vector_embed_folder_path, vector_embed_file_name)

        # Check if the output file already exists
        if os.path.exists(vector_embed_file_path):
            print(f"Skipping: {file_name} (Output already exists at {vector_embed_file_name})")
            continue  # Skip to the next file in the directory

        jsonl_file_path = os.path.join(jsonl_folder_path, file_name)

        try:
            with open(jsonl_file_path, 'r', encoding='utf-8') as f_in, \
                 open(vector_embed_file_path, 'w', encoding='utf-8') as f_out:

                batch_chunks = []
                batch_texts = []

                for line in f_in:
                    data = json.loads(line)
                    batch_chunks.append(data)
                    batch_texts.append(data['text'])

                    if len(batch_texts) == BATCH_SIZE:
                        embeddings = model.encode(
                            batch_texts,
                            batch_size=BATCH_SIZE,
                            show_progress_bar=False,
                            normalize_embeddings=True
                        )

                        for i, chunk in enumerate(batch_chunks):
                            vector_payload = {
                                "chunk_id": chunk["chunk_id"],
                                "vector": embeddings[i].tolist()
                            }
                            f_out.write(json.dumps(vector_payload) + '\n')

                        batch_chunks.clear()
                        batch_texts.clear()

                # Handle the last remaining partial batch
                if batch_texts:
                    embeddings = model.encode(
                        batch_texts,
                        batch_size=len(batch_texts),
                        show_progress_bar=False,
                        normalize_embeddings=True
                    )
                    for i, chunk in enumerate(batch_chunks):
                        vector_payload = {
                            "chunk_id": chunk["chunk_id"],
                            "vector": embeddings[i].tolist()
                        }
                        f_out.write(json.dumps(vector_payload) + '\n')

            print(f"Successfully processed and saved: {vector_embed_file_name}")

        except Exception as e:
            print(f"Severe error during converting text file in chunks from {file_name}: {e}")



Successfully processed and saved: 64data_vector.jsonl
Skipping: 29data.jsonl (Output already exists at 29data_vector.jsonl)
Skipping: 41data.jsonl (Output already exists at 41data_vector.jsonl)
Successfully processed and saved: 52data_vector.jsonl
Successfully processed and saved: 1data_vector.jsonl
Successfully processed and saved: 25data_vector.jsonl
Skipping: 5data.jsonl (Output already exists at 5data_vector.jsonl)
Successfully processed and saved: 54data_vector.jsonl
Successfully processed and saved: 39data_vector.jsonl
Successfully processed and saved: 28data_vector.jsonl
Skipping: 79data.jsonl (Output already exists at 79data_vector.jsonl)
Skipping: 97data.jsonl (Output already exists at 97data_vector.jsonl)
Successfully processed and saved: 89data_vector.jsonl
Successfully processed and saved: 43data_vector.jsonl
Successfully processed and saved: 15data_vector.jsonl
Successfully processed and saved: 69data_vector.jsonl
Successfully processed and saved: 99data_vector.jsonl
Succe